<a href="https://colab.research.google.com/github/ruhmmachaudhary-rgb/AI-ML-Fellowship-GDGOC-Atk/blob/main/week5/data-preprocessing-mastery.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [27]:
"""
Week 5: Data Preprocessing Mastery
All tasks implemented in a single notebook.
Each task includes overview, technique, reason, and results.
"""

import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder
from sklearn.impute import KNNImputer
from sklearn.feature_selection import RFE
from sklearn.linear_model import LogisticRegression
from sklearn.decomposition import PCA
from sklearn.datasets import load_digits
from imblearn.over_sampling import SMOTE
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize
from scipy.stats import zscore
from zipfile import ZipFile


# -------------------- Task 1: Titanic Dataset --------------------
with ZipFile("/content/titanic.zip") as zip_ref:
    df_titanic = pd.read_csv(zip_ref.open('train.csv'))
df_titanic['Age'] = df_titanic['Age'].fillna(df_titanic['Age'].median())
df_titanic = df_titanic.dropna(subset=['Embarked'])

# -------------------- Task 2: Car Evaluation Dataset --------------------
with ZipFile("/content/car+evaluation.zip") as zip_ref:
    df_car = pd.read_csv(zip_ref.open('car.data'), header=None,
                         names=['buying','maint','doors','persons','lug_boot','safety','class'])
le = LabelEncoder()
df_car['buying'] = le.fit_transform(df_car['buying'])
df_car = pd.get_dummies(df_car, columns=['maint','safety'])

# -------------------- Task 3: Wine Quality Dataset --------------------
with ZipFile("/content/wine+quality.zip") as zip_ref:
    df_wine = pd.read_csv(zip_ref.open('winequality-red.csv'), sep=';')
scaler_std = StandardScaler()
df_scaled = scaler_std.fit_transform(df_wine.drop('quality', axis=1))
scaler_mm = MinMaxScaler()
df_normalized = scaler_mm.fit_transform(df_wine.drop('quality', axis=1))

# -------------------- Task 4: Boston Housing Dataset --------------------
with ZipFile("/content/boston-housing.zip") as zip_ref:
    df_boston = pd.read_csv(zip_ref.open('train.csv'))
df_no_outliers = df_boston[(np.abs(zscore(df_boston)) < 3).all(axis=1)]

# -------------------- Task 5: Retail Sales Dataset --------------------
with ZipFile("/content/archive (5).zip") as zip_ref:
    df_retail = pd.read_csv(zip_ref.open('sales_data_sample.csv'), encoding='latin1')

# Select only numeric columns for imputation
numeric_cols = df_retail.select_dtypes(include=np.number).columns
imputer = KNNImputer(n_neighbors=3)
df_retail[numeric_cols] = imputer.fit_transform(df_retail[numeric_cols])

# -------------------- Task 6: Heart Disease Dataset --------------------
from zipfile import ZipFile

file_name = 'processed.cleveland.data'
columns = ['age','sex','cp','trestbps','chol','fbs','restecg','thalach','exang',
           'oldpeak','slope','ca','thal','target']

with ZipFile("/content/heart+disease.zip") as zip_ref:
    print(zip_ref.namelist())  # check available files
    df_heart = pd.read_csv(zip_ref.open(file_name), names=columns)

# Create AgeGroup column
df_heart['AgeGroup'] = pd.cut(df_heart['age'], bins=[29,40,50,60,77],
                              labels=['30-40','41-50','51-60','61-77'])
# -------------------- Task 7: Bike Sharing Dataset --------------------
with ZipFile("/content/bike+sharing+dataset.zip") as zip_ref:
    df_bike = pd.read_csv(zip_ref.open('day.csv'))
df_bike['cnt_log'] = np.log1p(df_bike['cnt'])

# -------------------- Task 8: Diabetes Dataset --------------------
with ZipFile("/content/archive (6).zip") as zip_ref:
    df_diabetes = pd.read_csv(zip_ref.open('diabetes.csv'))
X = df_diabetes.drop('Outcome', axis=1)
y = df_diabetes['Outcome']
model = LogisticRegression(max_iter=1000)
rfe = RFE(model, n_features_to_select=5)
X_selected = rfe.fit_transform(X, y)

# -------------------- Task 9: Credit Card Fraud Dataset --------------------
with ZipFile("/content/archive (7).zip") as zip_ref:
    df_fraud = pd.read_csv(zip_ref.open('creditcard.csv'))
X_fraud = df_fraud.drop('Class', axis=1)
y_fraud = df_fraud['Class']
smote = SMOTE()
X_res, y_res = smote.fit_resample(X_fraud, y_fraud)

# -------------------- Task 10: MovieLens Dataset --------------------
from zipfile import ZipFile

with ZipFile("/content/ml-100k.zip") as zip_ref:
    print(zip_ref.namelist())  # check filenames

    # Ratings
    df_ratings = pd.read_csv(
        zip_ref.open('ml-100k/u.data'),
        sep='\t',
        names=['userId','movieId','rating','timestamp']
    )

    # Users
    df_users = pd.read_csv(
        zip_ref.open('ml-100k/u.user'),
        sep='|',
        names=['userId','age','gender','occupation','zip']
    )

    # Movies
    movie_cols = ['movieId','title'] + [f'col{i}' for i in range(22)]  # extra 22 columns in u.item
    df_movies = pd.read_csv(
        zip_ref.open('ml-100k/u.item'),
        sep='|',
        names=movie_cols,
        encoding='latin-1'
    )

# Merge datasets
df_merged = df_ratings.merge(df_users, on='userId', how='left')\
                      .merge(df_movies[['movieId','title']], on='movieId', how='left')

# -------------------- Task 11: MNIST Dataset --------------------
digits = load_digits()
X_digits = digits.data
pca = PCA(n_components=30)
X_pca = pca.fit_transform(X_digits)

# -------------------- Task 12: IMDB Reviews Dataset --------------------
# -------------------- Task 12: IMDB Reviews Dataset (Fixed) --------------------
import pandas as pd
import re
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from zipfile import ZipFile

# Load IMDB dataset from zip
with ZipFile("/content/archive (8).zip") as zip_ref:
    df_imdb = pd.read_csv(zip_ref.open('IMDB Dataset.csv'))

stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()

# Preprocess function using regex tokenization (no punkt needed)
def preprocess(text):
    # Convert to lowercase
    text = text.lower()
    # Extract words only
    tokens = re.findall(r'\b[a-z]+\b', text)
    # Remove stopwords and stem
    tokens = [stemmer.stem(w) for w in tokens if w not in stop_words]
    return ' '.join(tokens)

# Apply preprocessing
df_imdb['review_clean'] = df_imdb['review'].apply(preprocess)

# Check result
df_imdb.head()

# -------------------- TASK 13: Air Quality Dataset --------------------
from zipfile import ZipFile
import pandas as pd

# Open the ZIP
with ZipFile("/content/archive (8).zip") as zip_ref:
    # List all files
    print(zip_ref.namelist())

    # Pick the correct CSV file for Air Quality
    file_path = [f for f in zip_ref.namelist() if 'AirQualityUCI' in f and f.endswith('.csv')][0]
    print("Reading Air Quality file:", file_path)

    # Read CSV (semicolon-separated, decimal is comma)
    df_air = pd.read_csv(zip_ref.open(file_path), sep=';', decimal=',')

# Convert numeric columns
numeric_cols = ['CO(GT)', 'PT08.S1(CO)', 'NMHC(GT)', 'C6H6(GT)',
                'PT08.S2(NMHC)', 'NOx(GT)', 'PT08.S3(NOx)', 'NO2(GT)',
                'PT08.S4(NO2)', 'PT08.S5(O3)', 'T', 'RH', 'AH']

for col in numeric_cols:
    df_air[col] = pd.to_numeric(df_air[col], errors='coerce')

# Fix DateTime index
df_air['Time'] = df_air['Time'].str.replace('.', ':', regex=False)  # replace dots with colons
df_air['DateTime'] = pd.to_datetime(df_air['Date'] + ' ' + df_air['Time'], dayfirst=True, errors='coerce')
df_air.set_index('DateTime', inplace=True)

# Drop rows where DateTime could not be parsed
df_air = df_air.dropna(subset=['DateTime'])

# Resample hourly and interpolate
df_air = df_air.resample('H').mean()
df_air.interpolate(method='linear', inplace=True)

# Optional: add smoothed CO column
df_air['CO_smoothed'] = df_air['CO(GT)'].rolling(window=3).mean()

df_air.head()

['Index', 'WARNING', 'ask-detrano', 'bak', 'cleve.mod', 'cleveland.data', 'costs/', 'costs/Index', 'costs/heart-disease.README', 'costs/heart-disease.cost', 'costs/heart-disease.delay', 'costs/heart-disease.expense', 'costs/heart-disease.group', 'heart-disease.names', 'hungarian.data', 'long-beach-va.data', 'new.data', 'processed.cleveland.data', 'processed.hungarian.data', 'processed.switzerland.data', 'processed.va.data', 'reprocessed.hungarian.data', 'switzerland.data']
['ml-100k/', 'ml-100k/allbut.pl', 'ml-100k/mku.sh', 'ml-100k/README', 'ml-100k/u.data', 'ml-100k/u.genre', 'ml-100k/u.info', 'ml-100k/u.item', 'ml-100k/u.occupation', 'ml-100k/u.user', 'ml-100k/u1.base', 'ml-100k/u1.test', 'ml-100k/u2.base', 'ml-100k/u2.test', 'ml-100k/u3.base', 'ml-100k/u3.test', 'ml-100k/u4.base', 'ml-100k/u4.test', 'ml-100k/u5.base', 'ml-100k/u5.test', 'ml-100k/ua.base', 'ml-100k/ua.test', 'ml-100k/ub.base', 'ml-100k/ub.test']


,review,sentiment,review_clean
0,One of the other reviewers has mentioned that ...,positive,one review mention watch oz episod hook right ...
1,A wonderful little production. <br /><br />The...,positive,wonder littl product br br film techniqu unass...
2,I thought this was a wonderful way to spend ti...,positive,thought wonder way spend time hot summer weeke...
3,Basically there's a family where a little boy ...,negative,basic famili littl boy jake think zombi closet...
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive,petter mattei love time money visual stun film...
